In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Hash Tables (Open Hashing)

This notebook provides a simple implementation of a [hash table](https://en.wikipedia.org/wiki/Hash_table) that uses *open hashing*.

The class `ListMap` from the notebook `ListMap.ipynb` implements a map as a *linked list*.

In [ ]:
import { ListMap } from './ListMap';

The function `charCodeAt(0)` maps characters to their [ASCII code](https://en.wikipedia.org/wiki/ASCII). $\mapsto$

In [ ]:
for (const c of "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ") {
  console.log(`${c} -> ${c.charCodeAt(0)}`);
}

Given a string $w$ and the size $n$ of the hash table, the function $\mathrm{hash\_code}(w,n)$ calculates the hash code of $w$. For a string
$w = c_0 c_1 \cdots c_{m-1}$ of length $m$, this function is defined as follows:
$$
\mathrm{hash\_code}(w,n)
= \left(\sum_{i=0}^{m-1} \operatorname{ord}(c_i)\cdot 128^i\right) \bmod n .
$$

In order to prevent overflows when computing the numbers $128^i$ we can define the partial sum $s_k$ for
$k = 0,1,\cdots,m-1$ by induction:
- $s_0 = \operatorname{ord}(c_{m-1}) \bmod n$,
- $s_{k+1} = \bigl(s_k \cdot 128 + \operatorname{ord}(c_k)\bigr) \bmod n$.

Then we have
$$
s_{m-1} = \left(\sum_{i=0}^{m-1} \operatorname{ord}(c_i)\cdot 128^i\right) \bmod n .
$$

In [ ]:
function hashCode(w: string, n: number): number {
  let s = 0;
  for (let i = w.length - 1; i >= 0; i--) {
    s = (s * 128 + w.charCodeAt(i)) % n;
  }
  return s;
}

Let us test this function.

In [ ]:
hashCode("George W. Bush", 6761);

In [ ]:
const HashTable = function <K extends string | number, V>(
  this: HashTable<K, V>,
  n: number,
  code: (key: K, m: number) => number
) {
  this.mSize = n;
  this.mEntries = 0;
  this.mArray = Array.from({ length: n }, () => new ListMap<K, V>());
  this.mAlpha = 2.0;
  this.mCode = code;
} as unknown as { new <K extends string | number, V>(n: number, code: (key: K, m: number) => number): HashTable<K, V> } & {
  Primes: number[];
};

interface HashTable<K extends string | number, V> {
  mSize: number;
  mEntries: number;
  mArray: Array<ListMap<K, V>>;
  mAlpha: number;
  mCode: (key: K, n: number) => number;

  find(key: K): V | null;
  insert(key: K, value: V): void;
  delete(key: K): boolean;
  allKeys(): K[];
  toString(): string;

  _rehash(grow?: boolean): void;
}

Hash tables work best if their size is a prime number.  Therefore, the variable `Primes` stores a list of prime numbers.  
These numbers are organized so that `Primes[i+1]` is roughly twice as big as `Primes[i]`.

In [ ]:
HashTable.Primes = [
  3, 7, 13, 31, 61, 127, 251, 509, 1021, 2039, 4093,
  8191, 16381, 32749, 65521, 131071, 262139, 524287,
  1048573, 2097143, 4194301, 8388593, 16777213,
  33554393, 67108859, 134217689, 268435399,
  536870909, 1073741789, 2147483647
];

In [ ]:
(HashTable as any).prototype.find = function <K extends string | number, V>(
  this: HashTable<K, V>,
  key: K
): V | null {
  const idx = ((this.mCode(key, this.mSize) % this.mSize) + this.mSize) % this.mSize;
  return this.mArray[idx].find(key);
};

In [ ]:
(HashTable as any).prototype.insert = function <K extends string | number, V>(
  this: HashTable<K, V>,
  key: K,
  value: V
): void {
  if (this.mEntries >= this.mSize * this.mAlpha) {
    this._rehash(true);
  }
  const idx = ((this.mCode(key, this.mSize) % this.mSize) + this.mSize) % this.mSize;
  const bucket = this.mArray[idx];
  const isNew = bucket.insert(key, value);
  if (isNew) this.mEntries++;
};


In [ ]:
(HashTable as any).prototype._rehash = function <K extends string | number, V>(
  this: HashTable<K, V>,
  grow: boolean = true
): void {
  const primes = (HashTable as any).Primes as number[];
  let next = this.mSize;

  if (grow) {
    for (const p of primes) {
      if (p > this.mSize && p * this.mAlpha > this.mEntries) { next = p; break; }
    }
  } else {
    for (let i = primes.length - 1; i >= 0; i--) {
      const p = primes[i];
      if (p < this.mSize && p * this.mAlpha >= this.mEntries) { next = p; break; }
    }
  }
  if (next === this.mSize) return;

  const newTable = new (HashTable as any)(next, this.mCode) as HashTable<K, V>;
  for (const bucket of this.mArray) {
    for (const [k, v] of bucket) newTable.insert(k, v);
  }
  this.mSize = newTable.mSize;
  this.mArray = newTable.mArray;
  this.mEntries = newTable.mEntries;
};

In [ ]:
(HashTable as any).prototype.delete = function <K extends string | number, V>(
  this: HashTable<K, V>,
  key: K
): boolean {
  if (2 * this.mEntries <= this.mSize * this.mAlpha) {
    this._rehash(false);
  }
  const idx = ((this.mCode(key, this.mSize) % this.mSize) + this.mSize) % this.mSize;
  const removed = this.mArray[idx].delete(key);
  if (removed) this.mEntries--;
  return removed;
};

In [ ]:
(HashTable as any).prototype.allKeys = function <K extends string | number, V>(
  this: HashTable<K, V>
): K[] {
  const out: K[] = [];
  for (const L of this.mArray) for (const [k] of L) out.push(k);
  return out;
};

In [ ]:
(HashTable as any).prototype.toString = function <K extends string | number, V>(
  this: HashTable<K, V>
): string {
  let s = "";
  for (let i = 0; i < this.mArray.length; i++) {
    s += `${i}: ${this.mArray[i].toString()}\n`;
  }
  return s;
};

In [ ]:
const t = new (HashTable as any)(3, hashCode) as HashTable<string, number>;

t.insert("Adrian", 8);
t.toString();

In [ ]:
t.insert('Benjamin', 24);
t.toString();

In [ ]:
t.insert('Bereket', 1);
t.toString();

In [ ]:
t.insert('Christian', 13);
t.toString();

In [ ]:
t.insert('Christian', 14);
t.toString();

In [ ]:
[ t.find("Adrian"), t.find("Christian"), t.find("Benjamin") ]

In [ ]:
t.insert('David', 22);
t.toString();

In [ ]:
t.insert('Ephraim', 19);
t.toString();

In [ ]:
t.insert('Erwin', 26);
t.toString();

In [ ]:
t.insert('Felix', 4);
t.toString();

In [ ]:
t.insert('Florian', 9);
t.toString();

In [ ]:
t.insert('Giorgio', 20);
t.toString();

In [ ]:
t.insert('Jan', 7);
t.toString();

In [ ]:
t.insert('Janis', 16);
t.toString();

In [ ]:
t.insert('Josia', 18);
t.toString();

In [ ]:
t.insert('Kai', 3);
t.toString();

In [ ]:
t.insert('Lars', 21);
t.toString();

In [ ]:
t.insert('Lucas', 0);
t.toString();

In [ ]:
t.insert('Marcel', 5);
t.toString();

In [ ]:
t.insert('Marius', 6);
t.toString();

In [ ]:
t.insert('Markus', 17);
t.toString();

In [ ]:
t.insert('Matthias', 10);
t.toString();

In [ ]:
t.insert('Nick', 11);
t.toString();

In [ ]:
t.insert('Patrick', 23);
t.toString();

In [ ]:
t.insert('Petra', 27);
t.toString();

In [ ]:
t.insert('Rene', 15);
t.toString();

In [ ]:
t.insert('Sebastian', 25);
t.toString();

In [ ]:
t.insert('Stefan', 2);
t.toString();

In [ ]:
t.find('Stefan');

In [ ]:
t.delete('Adrian');
t.toString();

In [ ]:
t.delete('Adrian');
t.delete('Benjamin');
t.delete('Bereket');
t.delete('Christian');
t.delete('Christian');
t.delete('David');
t.delete('Ephraim');
t.delete('Erwin');
t.delete('Felix');
t.delete('Florian');
t.delete('Giorgio');
t.delete('Jan');
t.delete('Janis');
t.delete('Josia');
t.delete('Kai');
t.delete('Lars');
t.delete('Lucas');
t.delete('Marcel');
t.delete('Marius');
t.delete('Markus');
t.delete('Matthias');
t.delete('Nick');
t.delete('Patrick');
t.delete('Petra');
t.delete('Rene');
t.delete('Sebastian');
t.delete('Stefan');
t.toString();

In [ ]:
function primes(n = 100): number[] {
  const S = new HashTable<number, number>(20, (x, m) => x % m);

  for (let i = 2; i <= n; i++) S.insert(i, i);

  for (let i = 2; i <= Math.floor(n / 2); i++) {
    for (let j = i; j <= Math.floor(n / i); j++) {
      S.delete(i * j);
    }
  }
  return S.allKeys().sort((a, b) => a - b);
}

In [ ]:
const L = primes();
console.log(L);